# Topic Modeling with BERTopic — Parameter Search & Evaluation

Loads pre-computed embeddings for the **iGEM teams** and **SynBio papers** datasets,
runs a grid search over key UMAP/HDBSCAN parameters, evaluates each configuration
using **topic coherence** ($C_v$), **topic diversity**, and **DBCV**, picks the best
model per dataset, and saves the results.

In [ ]:
import os
from itertools import product

import numpy as np
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic import BERTopic

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
ASSETS_DIR     = os.path.join('..', 'assets')
EMBEDDINGS_DIR = os.path.join(ASSETS_DIR, 'embeddings')
MODELS_DIR     = os.path.join(ASSETS_DIR, 'topic_models')
os.makedirs(MODELS_DIR, exist_ok=True)

# ── PARAMETER GRID ────────────────────────────────────────────────────────────
PARAM_GRID_TEAMS = {
    'min_cluster_size': [8, 10, 15, 20],
    'umap_n_neighbors': [10, 15, 25],
    'umap_n_components': [5, 10],
}

PARAM_GRID_PAPERS = {
    'min_cluster_size': [20, 30, 40, 50],
    'umap_n_neighbors': [10, 15, 25],
    'umap_n_components': [5, 10],
}

## 1. Load embeddings and corpora

In [ ]:
teams_embeddings  = np.load(os.path.join(EMBEDDINGS_DIR, 'teams_embeddings.npy'))
papers_embeddings = np.load(os.path.join(EMBEDDINGS_DIR, 'papers_embeddings.npy'))

teams_corpus  = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'teams_corpus.txt'), sep='\t')
papers_corpus = pd.read_csv(os.path.join(EMBEDDINGS_DIR, 'papers_corpus.txt'), sep='\t')

teams_docs  = teams_corpus['text'].tolist()
papers_docs = papers_corpus['text'].tolist()

print(f'Teams:  {teams_embeddings.shape[0]:,} docs, {teams_embeddings.shape[1]} dims')
print(f'Papers: {papers_embeddings.shape[0]:,} docs, {papers_embeddings.shape[1]} dims')

assert len(teams_docs) == teams_embeddings.shape[0]
assert len(papers_docs) == papers_embeddings.shape[0]

## 2. Helpers: model fitting and evaluation metrics

In [ ]:
def fit_topic_model(
    docs: list[str],
    embeddings: np.ndarray,
    min_cluster_size: int = 15,
    umap_n_neighbors: int = 15,
    umap_n_components: int = 5,
) -> tuple[BERTopic, list[int], np.ndarray]:
    """Fit a BERTopic model using UMAP + HDBSCAN."""
    umap_model = UMAP(
        n_neighbors=umap_n_neighbors,
        n_components=umap_n_components,
        min_dist=0.0,
        metric='cosine',
        random_state=42,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True,
    )
    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        min_topic_size=min_cluster_size,
        n_gram_range=(1, 3),
        language='english',
        calculate_probabilities=True,
        verbose=False,
    )
    topics, probs = topic_model.fit_transform(docs, embeddings)
    return topic_model, topics, probs

In [ ]:
def coherence_cv(topic_model: BERTopic, docs: list[str], top_n: int = 10) -> float:
    """Compute C_v coherence over the top-N words of each non-outlier topic."""
    topics = topic_model.get_topics()
    topic_words = [
        [word for word, _ in topics[t][:top_n]]
        for t in sorted(topics) if t != -1
    ]
    if not topic_words:
        return float('nan')
    tokenized = [doc.lower().split() for doc in docs]
    dictionary = Dictionary(tokenized)
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized,
        dictionary=dictionary,
        coherence='c_v',
    )
    return cm.get_coherence()


def topic_diversity(topic_model: BERTopic, top_n: int = 10) -> float:
    """Fraction of unique words across all topics' top-N word lists."""
    topics = topic_model.get_topics()
    all_words = [
        word
        for t in sorted(topics) if t != -1
        for word, _ in topics[t][:top_n]
    ]
    return len(set(all_words)) / len(all_words) if all_words else float('nan')


def dbcv_score(topic_model: BERTopic) -> float:
    """HDBSCAN relative validity (DBCV). Higher is better, range [-1, 1]."""
    return topic_model.hdbscan_model.relative_validity_

In [ ]:
def grid_search(
    docs: list[str],
    embeddings: np.ndarray,
    param_grid: dict,
    label: str = '',
) -> pd.DataFrame:
    """Run all parameter combinations, evaluate each, return results DataFrame."""
    combos = list(product(
        param_grid['min_cluster_size'],
        param_grid['umap_n_neighbors'],
        param_grid['umap_n_components'],
    ))
    n_total = len(combos)
    rows = []
    best_score = -np.inf
    best_model_data = None

    for i, (mcs, nn, nc) in enumerate(combos, 1):
        print(f'[{label}] {i}/{n_total}  mcs={mcs}, nn={nn}, nc={nc} ... ', end='', flush=True)
        model, topics, probs = fit_topic_model(
            docs, embeddings,
            min_cluster_size=mcs, umap_n_neighbors=nn, umap_n_components=nc,
        )
        n_topics = model.get_topic_info().Topic.max() + 1
        outlier_frac = (np.array(topics) == -1).mean()
        cv  = coherence_cv(model, docs)
        div = topic_diversity(model)
        dbcv = dbcv_score(model)

        row = {
            'min_cluster_size': mcs,
            'n_neighbors': nn,
            'n_components': nc,
            'n_topics': n_topics,
            'outlier_frac': round(outlier_frac, 4),
            'coherence_cv': round(cv, 4),
            'diversity': round(div, 4),
            'dbcv': round(dbcv, 4),
        }
        rows.append(row)
        print(f'topics={n_topics}, C_v={cv:.4f}, div={div:.4f}, DBCV={dbcv:.4f}')

        # Track best model by C_v (primary), break ties by diversity
        if cv > best_score or (cv == best_score and div > (best_model_data or {}).get('diversity', -1)):
            best_score = cv
            best_model_data = {
                'model': model, 'topics': topics, 'probs': probs,
                'diversity': div, **row,
            }

    df = pd.DataFrame(rows).sort_values('coherence_cv', ascending=False).reset_index(drop=True)
    return df, best_model_data

## 3. Grid search: iGEM Teams

In [ ]:
teams_results, teams_best = grid_search(
    teams_docs, teams_embeddings, PARAM_GRID_TEAMS, label='Teams'
)
teams_results

In [ ]:
print('Best Teams configuration:')
print(f"  min_cluster_size = {teams_best['min_cluster_size']}")
print(f"  n_neighbors      = {teams_best['n_neighbors']}")
print(f"  n_components     = {teams_best['n_components']}")
print(f"  n_topics         = {teams_best['n_topics']}")
print(f"  C_v              = {teams_best['coherence_cv']}")
print(f"  diversity        = {teams_best['diversity']}")
print(f"  DBCV             = {teams_best['dbcv']}")
print(f"  outlier_frac     = {teams_best['outlier_frac']}")

## 4. Grid search: SynBio Papers

In [ ]:
papers_results, papers_best = grid_search(
    papers_docs, papers_embeddings, PARAM_GRID_PAPERS, label='Papers'
)
papers_results

In [ ]:
print('Best Papers configuration:')
print(f"  min_cluster_size = {papers_best['min_cluster_size']}")
print(f"  n_neighbors      = {papers_best['n_neighbors']}")
print(f"  n_components     = {papers_best['n_components']}")
print(f"  n_topics         = {papers_best['n_topics']}")
print(f"  C_v              = {papers_best['coherence_cv']}")
print(f"  diversity        = {papers_best['diversity']}")
print(f"  DBCV             = {papers_best['dbcv']}")
print(f"  outlier_frac     = {papers_best['outlier_frac']}")

## 5. Save best models and outputs

In [ ]:
teams_model  = teams_best['model']
papers_model = papers_best['model']
teams_topics  = teams_best['topics']
papers_topics = papers_best['topics']

# ── Save models ───────────────────────────────────────────────────────────────
teams_model.save(os.path.join(MODELS_DIR, 'teams_topic_model'))
papers_model.save(os.path.join(MODELS_DIR, 'papers_topic_model'))

# ── Save topic info summaries ─────────────────────────────────────────────────
teams_info  = teams_model.get_topic_info()
papers_info = papers_model.get_topic_info()

teams_info.to_csv(os.path.join(MODELS_DIR, 'teams_topic_info.txt'), sep='\t', index=False)
papers_info.to_csv(os.path.join(MODELS_DIR, 'papers_topic_info.txt'), sep='\t', index=False)

# ── Save document-level topic assignments ─────────────────────────────────────
teams_corpus['topic'] = teams_topics
teams_corpus[['UT', 'topic']].to_csv(
    os.path.join(MODELS_DIR, 'teams_doc_topics.txt'), sep='\t', index=False
)
papers_corpus['topic'] = papers_topics
papers_corpus[['id', 'topic']].to_csv(
    os.path.join(MODELS_DIR, 'papers_doc_topics.txt'), sep='\t', index=False
)

# ── Save grid search results ─────────────────────────────────────────────────
teams_results.to_csv(os.path.join(MODELS_DIR, 'teams_grid_search.txt'), sep='\t', index=False)
papers_results.to_csv(os.path.join(MODELS_DIR, 'papers_grid_search.txt'), sep='\t', index=False)

print(f'All outputs saved to {MODELS_DIR}')
for f in sorted(os.listdir(MODELS_DIR)):
    print(f'  {f}')